# 🏦 Topic 1: Foundations - LLMs, APIs, and Tokens

Welcome to Day 1! Over the next three days you are going to build a production-ready
Barclays Product Knowledge Assistant - a chatbot that answers customer questions about
Barclays products. Today we lay the foundation.

## Learning Objectives

By the end of this topic you will be able to:

1. Explain what a large language model is and how it generates text one token at a time
2. Make a working API call to OpenAI GPT-4o from Python
3. Parse the response object to extract the answer, token counts, and cost estimate
4. Recognize the most common mistakes beginners make with the API

## What We Are Building Today

The core of the Barclays Product Knowledge Assistant: a single API call that takes a
customer question and a product snippet, and returns a grounded, helpful answer.

By end of Topic 1:

```
BEFORE -> customer asks a question, gets nothing
AFTER  -> customer asks a question, gets a clear answer powered by GPT-4o
```

In Topic 2 we will learn how to load real Barclays product documents so we do not need
to hardcode the product snippet. But first - let's get the API working.

## ⚙️ Section 0: Environment Setup

Let's install the packages we need and connect to AWS SageMaker and the OpenAI API.
This takes about 2-3 minutes the first time.

In [ ]:
# Install the OpenAI Python SDK (pinned to a stable version)
# numpy<2 is pinned to avoid compatibility issues with other SageMaker packages
# tiktoken is OpenAI's official token-counting library
# sagemaker==2.232.1 is the last classic SDK release where get_execution_role lives at the top level
# boto3 is the AWS SDK - needed for S3 and SageMaker session
!pip install -q "openai==2.32.0" "numpy<2" "tiktoken==0.9.0" "sagemaker==2.232.1" "boto3"

In [ ]:
import os
import getpass

from openai import OpenAI   # main synchronous client class
import tiktoken             # OpenAI's token-counting library

# Load the OpenAI API key securely - getpass hides what you type
# Stored as an environment variable so we can access it anywhere in the notebook
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key and press Enter: ")

# Create the OpenAI client - reads OPENAI_API_KEY from os.environ automatically
client = OpenAI()

MODEL = "gpt-4o"  # course-wide model constant

print("API key loaded. (Stored in memory only for this session.)")

## 🎯 What Are We Building Today?

Today I'm going to show you how to call a large language model API from Python and get
grounded, reliable answers about Barclays products.

Here is the before/after we are working toward:

**BEFORE** (no API call yet):
```python
customer_question = "What is the interest rate on a Barclays personal loan?"
answer = "???"   # we have no way to answer this yet
```

**AFTER** (by end of this topic):
```python
customer_question = "What is the interest rate on a Barclays personal loan?"
answer = ask_with_cost_tracking(customer_question, product_snippet)
# -> "Based on the product summary, the representative APR is 6.5% for
#     loans between 7,500 GBP and 15,000 GBP..."
```

We will build this in three steps:
1. Understand what an LLM is and how tokens work
2. Make a working API call with context injection
3. Read the response object and track cost

Let's go! 🚀

## 🧠 Concept 1: What is an LLM and How Do Tokens Work?

Before we write a single line of API code, I want to show you something that trips
up almost everyone the first time they work with an LLM.

Here is the problem. Watch what happens when we ask the model a product-specific question
without giving it any product data to work from.

In [ ]:
# NAIVE DEMO - this is what NOT to do
# Watch what happens when we call the API with no product data at all
# The model has no idea what Barclays currently offers - it can only guess

_demo_client = OpenAI()   # reads OPENAI_API_KEY from os.environ

naive_response = _demo_client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            # No product context, no system prompt - just a raw question
            "content": "What is the exact current interest rate on the Barclays personal loan?"
        }
    ]
)

print("Model answer (no product context given):")
print(naive_response.choices[0].message.content)
print()
print("Notice: the model hedges, may be out of date, or may hallucinate a number.")
print("This is the core problem we fix today - always inject product data into the prompt.")

### How Does an LLM Actually Work?

```mermaid
graph TD
    A["Input text<br/>'What is the APR?'"] --> B["Tokenizer<br/>splits into tokens"]
    B --> C["Token IDs<br/>[1867, 318, 262, 317, 4805, 30]"]
    C --> D["Transformer decoder<br/>GPT-4o"]
    D --> E["Next-token prediction<br/>probability over vocabulary"]
    E --> F["Sample or argmax<br/>controlled by temperature"]
    F --> G["New token appended<br/>to context"]
    G --> H{Stop condition?}
    H -->|No - keep generating| D
    H -->|Yes - stop token or max_tokens| I["Final response text<br/>assembled from all tokens"]
```

An LLM predicts the next token given all previous tokens - one step at a time. This is why
it cannot look up live data and why injecting product context into the prompt is essential.

## 🔌 Concept 2: Making Your First Real API Call

Now let's call the API properly - with a system prompt that sets context and a
product snippet injected into the user message.

First, let me show you the two most common mistakes beginners make when setting up the client.

In [ ]:
# BROKEN PATTERNS - do not do these
# Mistake 1: hardcoding the API key directly in the notebook
# Anyone who gets your notebook file gets your key - a security problem
try:
    broken_client = OpenAI(api_key="sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
    print("Client created with hardcoded key - but this is dangerous.")
    print("If this were a real key, sharing the notebook = sharing the key.")
except Exception as e:
    print(f"Error: {e}")

print()

# Mistake 2: forgetting to set the key at all before creating the client
import os as _os
saved_key = _os.environ.pop("OPENAI_API_KEY", None)  # temporarily remove to simulate mistake
try:
    no_key_client = OpenAI()   # will raise AuthenticationError - no key in env
    print("This would fail on the first real API call with AuthenticationError.")
except Exception as e:
    print(f"Error (no key set): {type(e).__name__}")
finally:
    if saved_key:
        _os.environ["OPENAI_API_KEY"] = saved_key   # restore key for rest of notebook

print()
print("The RIGHT pattern: set os.environ[\"OPENAI_API_KEY\"] once via getpass, then use OpenAI().")

### The Full API Round-Trip

```mermaid
sequenceDiagram
    participant P as Python client
    participant S as OpenAI SDK
    participant G as API gateway
    participant M as GPT-4o model
    P->>S: client.chat.completions.create(model, messages)
    S->>G: HTTPS POST /v1/chat/completions + API key
    G->>G: Authenticate key + route request
    G->>M: Tokenized prompt
    M->>M: Generate tokens one at a time
    M->>G: Token stream
    G->>S: JSON response object
    S->>P: ChatCompletion object
    Note over P: response.choices[0].message.content
    Note over P: response.usage.prompt_tokens
    Note over P: response.usage.completion_tokens
    Note over P: response.choices[0].finish_reason
```

When you call `client.chat.completions.create(...)`, your code sends a JSON payload over HTTPS,
the model generates tokens one at a time, and the API returns a structured response object
with the answer text plus usage metadata you can use for cost tracking.

In [ ]:
# FULL WORKING DEMO - the correct pattern for the entire course
# I'm going to show you the exact call we will build on in every topic from here.

# Step 1: define a hardcoded product snippet
# In Topic 2 we load this from real Barclays PDFs; for now we focus on the API mechanics
PRODUCT_SNIPPET = """
Barclays Personal Loan - Product Summary (illustrative)
- Loan amounts: 1,000 GBP to 35,000 GBP
- Representative APR: 6.5% for loans of 7,500 GBP to 15,000 GBP
- Repayment terms: 1 to 5 years
- No arrangement fee
- Early repayment allowed with 30-day interest charge
"""

# Step 2: build the messages list - the core of the Chat Completions API
# "system" sets the assistant's persona and rules
# "user" carries the customer's question plus the product context
customer_question = "What is the interest rate on the Barclays personal loan?"

messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful Barclays product knowledge assistant. "
            "Answer questions using only the product information provided. "
            "Be concise and factual. Do not invent numbers."
        )
    },
    {
        "role": "user",
        # We inject the product snippet directly into the user message
        # This is the simplest form of context injection - Topics 6-7 do this via RAG
        "content": f"Product information:\n{PRODUCT_SNIPPET}\n\nCustomer question: {customer_question}"
    }
]

# Step 3: call the API - MODEL is defined in Cell 3 as "gpt-4o"
response = client.chat.completions.create(
    model=MODEL,          # course-wide model constant
    messages=messages,
    temperature=0.2,      # low temperature = consistent, factual answers
    max_tokens=300        # cap response length to control cost
)

# Step 4: extract the answer
answer = response.choices[0].message.content

print("Customer question:", customer_question)
print()
print("Assistant answer:")
print(answer)
print()
print("--- Response metadata ---")
print(f"Model used:        {response.model}")
print(f"Finish reason:     {response.choices[0].finish_reason}")
print(f"Prompt tokens:     {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")

## 🧪 Lab 1: Your First API Call

Now it's your turn. I want you to call the OpenAI API with a different Barclays product question.

**Your task**: Ask the assistant about a different aspect of the Barclays personal loan
(for example: repayment terms, early repayment rules, or loan amount limits).

**Stretch** (if you finish early): Change `temperature` to `0.9` and run the cell several
times. Does the answer change? Why do you think that is?

**Time**: 15-20 minutes

In [ ]:
# LAB 1 SOLUTION: Your First API Call

# Step 1: a different question about the personal loan - repayment terms this time
lab_question = "What repayment term options are available for the Barclays personal loan?"

# Step 2: build a messages list with system prompt and product context injected
# Same structure as the full demo - system role sets persona, user role carries context + question
lab_messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful Barclays product knowledge assistant. "
            "Answer using only the product information provided. Be concise and factual."
        )
    },
    {
        "role": "user",
        # Inject PRODUCT_SNIPPET so the model answers from real data, not guesswork
        "content": f"Product information:\n{PRODUCT_SNIPPET}\n\nCustomer question: {lab_question}"
    }
]

# Step 3: call the API - same model and temperature as the demo
lab_response = client.chat.completions.create(
    model="gpt-4o",
    messages=lab_messages,
    temperature=0.2,   # low temp = consistent, factual answers - good for Q&A
    max_tokens=300
)

# Step 4: pull the answer text out of the response object
lab_answer = lab_response.choices[0].message.content

print("Your question:", lab_question)
print()
print("Assistant answer:")
print(lab_answer)
print()
print("[OK] Lab 1 complete!")

# COMMON MISTAKES students make here:
# 1. Forgetting to include PRODUCT_SNIPPET in the user message -> model hallucinates numbers
# 2. Using response.content directly instead of response.choices[0].message.content
# 3. Setting temperature=1.0 for factual Q&A -> inconsistent answers across runs

In [ ]:
# Safety-net: if you did not finish Lab 1, run this cell so the rest of the
# notebook still works. If you DID finish Lab 1 above, skip this cell.
if lab_answer is None:
    print("Using safety-net so the notebook can continue.")
    _sn_msgs = [
        {"role": "system", "content": "You are a helpful Barclays product knowledge assistant. Answer using only the product information provided."},
        {"role": "user", "content": f"Product information:\n{PRODUCT_SNIPPET}\n\nCustomer question: What are the repayment term options?"}
    ]
    _sn_resp = client.chat.completions.create(model="gpt-4o", messages=_sn_msgs, temperature=0.2, max_tokens=200)
    lab_answer = _sn_resp.choices[0].message.content
    lab_question = "What are the repayment term options?"
    print("Safety-net answer:", lab_answer[:100], "...")

## 💰 Concept 3: Understanding the Response Object and Token Cost

Here is a mistake I see developers make in production all the time: they start calling
the API in a loop without tracking token usage. Then the monthly bill arrives and it's
much larger than expected.

Let me show you what happens when you ignore the token counts the API returns with every response.

In [ ]:
# NAIVE PATTERN - ignoring the response object beyond the answer text
# This works for a single call but is dangerous in any production loop

def naive_ask(question: str) -> str:
    """Ask a question, return only the text. No cost tracking at all."""
    r = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": question}],
        max_tokens=200
    )
    # We only look at the text - throwing away all the metadata
    return r.choices[0].message.content

sample_questions = [
    "What is a personal loan?",
    "How do I apply for a Barclays account?",
    "Can I repay a loan early?",
]

total_cost = 0.0   # this stays 0 because we are not tracking anything

for q in sample_questions:
    ans = naive_ask(q)
    print(f"Q: {q}")
    print(f"A: {ans[:80]}...")
    print()

print(f"Total estimated cost: ${total_cost:.4f}")
print("Problem: we have no idea what this actually cost - dangerous in production.")

### Token Counting and Cost Awareness

```mermaid
graph TD
    A["API call completes"] --> B["response.usage"]
    B --> C["prompt_tokens<br/>e.g. 320"]
    B --> D["completion_tokens<br/>e.g. 150"]
    C --> E["Input cost<br/>320 / 1000 x 0.0025<br/>= 0.000800 USD"]
    D --> F["Output cost<br/>150 / 1000 x 0.0100<br/>= 0.001500 USD"]
    E --> G["Total cost<br/>0.000800 + 0.001500<br/>= 0.002300 USD per call"]
    F --> G
    G --> H["At 1,000 calls/day<br/>= 2.30 USD/day<br/>= ~69 USD/month"]
```

Every API response includes `response.usage` with exact token counts. GPT-4o charges
$0.0025 per 1K input tokens and $0.0100 per 1K output tokens - tiny per call, but
at thousands of calls per day it adds up fast.

In [ ]:
# FULL WORKING DEMO - reading the response object and computing cost

# GPT-4o pricing as of April 2026
GPT4O_INPUT_PRICE_PER_1K  = 0.0025   # dollars per 1,000 input tokens
GPT4O_OUTPUT_PRICE_PER_1K = 0.0100   # dollars per 1,000 output tokens

def ask_with_cost_tracking(question: str, context: str) -> dict:
    """
    Ask a question with context injection. Returns answer plus usage stats.
    Keys: answer, prompt_tokens, completion_tokens, cost_usd, finish_reason
    """
    msgs = [
        {
            "role": "system",
            "content": "You are a helpful Barclays product knowledge assistant. Answer using only the provided product information."
        },
        {
            "role": "user",
            "content": f"Product information:\n{context}\n\nCustomer question: {question}"
        }
    ]

    r = client.chat.completions.create(
        model="gpt-4o",
        messages=msgs,
        temperature=0.2,
        max_tokens=300
    )

    prompt_tokens     = r.usage.prompt_tokens
    completion_tokens = r.usage.completion_tokens

    # Cost = tokens used / 1000 * price per 1K
    cost = (prompt_tokens / 1000) * GPT4O_INPUT_PRICE_PER_1K + \
           (completion_tokens / 1000) * GPT4O_OUTPUT_PRICE_PER_1K

    return {
        "answer":            r.choices[0].message.content,
        "prompt_tokens":     prompt_tokens,
        "completion_tokens": completion_tokens,
        "cost_usd":          cost,
        "finish_reason":     r.choices[0].finish_reason
    }

# Run a sample call
result = ask_with_cost_tracking(
    question="How much can I borrow with a Barclays personal loan?",
    context=PRODUCT_SNIPPET
)

print("Answer:", result["answer"])
print()
print("--- Token usage ---")
print(f"Prompt tokens:     {result['prompt_tokens']}")
print(f"Completion tokens: {result['completion_tokens']}")
print(f"Finish reason:     {result['finish_reason']}")
print(f"Estimated cost:    ${result['cost_usd']:.6f}")

# Bonus: pre-flight token count using tiktoken
print()
print("--- Pre-flight token estimate using tiktoken ---")
enc = tiktoken.encoding_for_model("gpt-4o")
sample = f"You are a helpful assistant.\n{PRODUCT_SNIPPET}\nHow much can I borrow?"
preflight = len(enc.encode(sample))
print(f"tiktoken estimate: {preflight} tokens")
print(f"Actual from API:   {result['prompt_tokens']} tokens")
print("(Small difference is normal - system overhead from the messages structure.)")

## 🧪 Lab 2: Token Counting and Cost Estimation

Now it's your turn to build a cost-tracking batch function.

**Your task**: Complete `estimate_batch_cost` below so it calls the API for each question,
accumulates the total cost, and returns it.

**Stretch** (if you finish early): Use tiktoken to check what happens to the token count
when you double the length of `PRODUCT_SNIPPET`. Is the relationship linear?

**Time**: 15-20 minutes

In [ ]:
# LAB 2 SOLUTION: Token Counting and Cost Estimation

SAMPLE_QUESTIONS = [
    "What loan amounts are available?",
    "Is there an arrangement fee?",
    "Can I repay the loan early?",
]

def estimate_batch_cost(questions: list, context: str) -> float:
    """
    Call ask_with_cost_tracking for each question and return total cost in USD.
    Print a summary line for each call showing token counts and cost.
    """
    # Step 1: initialise the running total at zero
    total_cost = 0.0

    for question in questions:
        # Step 2: call ask_with_cost_tracking - returns dict with answer, tokens, cost
        result = ask_with_cost_tracking(question, context)

        # Step 3: accumulate the cost for this call into the running total
        total_cost += result["cost_usd"]

        print(f"  Q: {question}")
        print(f"     tokens: {result['prompt_tokens']}p + {result['completion_tokens']}c  cost: ${result['cost_usd']:.6f}")

    return total_cost


# Step 4: run the batch over all sample questions using PRODUCT_SNIPPET as context
batch_total = estimate_batch_cost(SAMPLE_QUESTIONS, PRODUCT_SNIPPET)

print()
print(f"Total cost for {len(SAMPLE_QUESTIONS)} questions: ${batch_total:.6f}")
daily = batch_total / len(SAMPLE_QUESTIONS) * 1000
print(f"Rough daily estimate at 1,000 calls/day: ${daily:.4f}")
print("[OK] Lab 2 complete!")

# COMMON MISTAKES students make here:
# 1. Forgetting to initialise total_cost = 0.0 before the loop -> NameError or wrong sum
# 2. Using += result["cost"] instead of += result["cost_usd"] -> KeyError
# 3. Passing only the question to estimate_batch_cost without context -> missing argument

## ✅ Topic 1 Wrap-Up: What We Built Today

Here is what you built in this topic and how it fits into the bigger picture.

**What you built today**:
- A working OpenAI GPT-4o API call from Python in SageMaker
- Context injection: product data in the prompt so the model gives grounded answers
- `ask_with_cost_tracking()`: reads token usage from every response
- Pre-flight token estimation using tiktoken

**The Barclays Product Knowledge Assistant so far**:
```
customer_question
      |
      v
[inject PRODUCT_SNIPPET into prompt]
      |
      v
[GPT-4o via OpenAI API]
      |
      v
answer  +  token cost logged
```

**Key takeaways**:
- Always inject product/domain data - the model cannot look up live information
- Read `response.usage` after every call to track cost in production
- `finish_reason == "length"` means the answer was cut off - increase max_tokens
- Temperature 0.2 is a good default for factual Q&A

**What comes next - Topic 2: NLP Preprocessing for RAG**

Right now we hardcoded the product snippet. In Topic 2 we load real Barclays PDF documents,
clean the extracted text, and chunk it into pieces ready for the RAG pipeline in Topics 6-7.

**Homework extensions**:
1. Wrap `ask_with_cost_tracking` in a `try/except` that catches `openai.RateLimitError` and retries after 10 seconds
2. Add a `max_budget_usd` parameter to `estimate_batch_cost` that stops if the running total exceeds the budget
3. Use tiktoken to estimate the cost of embedding a 50-page PDF - what chunk size keeps each chunk under $0.001?